# Main Figure 1 - Synthetic benchmark

            This notebook is a cleaned, English-commented manuscript code file.

            **Provenance.** A100 `/data/taobo.hu/projects/mouse_pup/benchmarking_complete_english.ipynb`, `benchmarking_complete_modular.ipynb`, and `benchmarking_complete_nested.ipynb`.

            The notebook keeps the original manuscript data paths where they were found. If a path points to
            `/data/taobo.hu` or `/mnt/taobo.hu`, run the notebook on the A100 server or adapt the path to the
            matching mounted Y-drive location.


In [ ]:
# Panel mapping:
# - Figure 1a-i, shared setup. Imports the numerical, plotting, and table libraries used by every synthetic benchmark panel in this notebook.
#

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
sns.set_theme(context="talk", style="white")


In [ ]:
# Panel mapping:
# - Figure 1a-c and Supplementary Figures 1-3, data-generation helper. Defines the modular, nested, and random synthetic point patterns used as panel inputs before any method is run.
#

CELL_TYPES = list("ABCDEF")


def make_synthetic_modular(n_per_type=200, seed=42, spread=0.6, module12_dist=5.0):
    """Create the modular synthetic pattern: A/B, C/D, and E/F form three modules."""
    rng = np.random.default_rng(seed)
    centers = {
        "A": (0.0, 0.0),
        "B": (0.0, 0.0),
        "C": (module12_dist, 0.0),
        "D": (module12_dist, 0.0),
        "E": (module12_dist / 2.0, 5.0),
        "F": (module12_dist / 2.0, 5.0),
    }
    frames = []
    for label, (cx, cy) in centers.items():
        xy = rng.normal(loc=(cx, cy), scale=spread, size=(n_per_type, 2))
        frames.append(pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "celltype": label}))
    return pd.concat(frames, ignore_index=True)


def _annulus(rng, center, n, inner, outer):
    theta = rng.uniform(0, 2 * np.pi, n)
    radius = np.sqrt(rng.uniform(inner**2, outer**2, n))
    return np.column_stack([center[0] + radius * np.cos(theta), center[1] + radius * np.sin(theta)])


def make_nested_variant(
    n_core=150,
    n_middle=250,
    n_outer=400,
    center1=(0.0, 0.0),
    center2=(8.0, 8.0),
    radii=((0, 0.4), (0.5, 1.1), (1.2, 2.2)),
    seed=123,
):
    """Create nested A/B/C and D/E/F shell structures."""
    rng = np.random.default_rng(seed)
    specs = [
        ("A", center1, n_core, *radii[0]),
        ("B", center1, n_middle, *radii[1]),
        ("C", center1, n_outer, *radii[2]),
        ("D", center2, n_core, *radii[0]),
        ("E", center2, n_middle, *radii[1]),
        ("F", center2, n_outer, *radii[2]),
    ]
    frames = []
    for label, center, n, inner, outer in specs:
        xy = _annulus(rng, center, n, inner, outer)
        frames.append(pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "celltype": label}))
    return pd.concat(frames, ignore_index=True)


def make_synthetic_random(n_per_type=300, seed=42):
    """Create the well-mixed random negative-control pattern."""
    rng = np.random.default_rng(seed)
    frames = []
    for label in CELL_TYPES:
        xy = rng.uniform(-1, 9, size=(n_per_type, 2))
        frames.append(pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "celltype": label}))
    return pd.concat(frames, ignore_index=True)


def nested_dataset_collection():
    """Return the twelve nested perturbation datasets used in Supplementary Figure 2."""
    radius_conditions = [
        ((0, 0.3), (0.5, 1.0), (1.2, 2.0)),
        ((0, 0.4), (0.6, 1.2), (1.5, 2.6)),
        ((0, 0.5), (0.8, 1.6), (2.0, 3.2)),
        ((0, 0.7), (1.0, 2.0), (2.5, 4.0)),
    ]
    thickness_conditions = [
        ((0, 0.35), (0.45, 0.75), (0.85, 1.2)),
        ((0, 0.35), (0.55, 1.0), (1.2, 1.9)),
        ((0, 0.35), (0.65, 1.3), (1.6, 2.7)),
        ((0, 0.35), (0.80, 1.7), (2.1, 3.6)),
    ]
    center_conditions = [((0, 0), (5, 5)), ((0, 0), (7, 7)), ((0, 0), (9, 9)), ((0, 0), (11, 11))]
    out = {}
    for i, radii in enumerate(radius_conditions, 1):
        out[f"nested_radius_{i}"] = make_nested_variant(radii=radii, seed=100 + i)
    for i, radii in enumerate(thickness_conditions, 1):
        out[f"nested_thickness_{i}"] = make_nested_variant(radii=radii, seed=200 + i)
    for i, (center1, center2) in enumerate(center_conditions, 1):
        out[f"nested_centers_{i}"] = make_nested_variant(center1=center1, center2=center2, seed=300 + i)
    return out


In [ ]:
# Panel mapping:
# - Figure 1d-f, Cell-GPS/COSTE helper. Computes the Searcher's D / SSS StructureMap matrix that is rendered as the main COSTE heatmap panels.
#

from sfplot import compute_cophenetic_distances_from_df, plot_cophenetic_heatmap


def compute_coste_structuremap(points: pd.DataFrame, sample_name: str, output_dir: Path):
    """Compute the COSTE/Cell-GPS StructureMap and save the heatmap/table.

    The required table columns are `x`, `y`, and `celltype`. The manuscript uses the
    row cophenetic matrix as the Searcher's D / SSS heatmap.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    row_coph, col_coph = compute_cophenetic_distances_from_df(
        points[["x", "y", "celltype"]],
        x_col="x",
        y_col="y",
        celltype_col="celltype",
        method="average",
        show_corr=False,
    )
    row_coph.to_csv(output_dir / f"StructureMap_table_{sample_name}.csv")
    plot_cophenetic_heatmap(
        row_coph,
        matrix_name="row_coph",
        output_filename=str(output_dir / f"StructureMap_of_{sample_name}.pdf"),
        sample=sample_name,
    )
    return row_coph, col_coph


In [ ]:
# Panel mapping:
# - Figure 1a-c and Figure 1d-i, plotting helper. Saves the spatial point-cloud panels and method heatmaps with consistent manuscript styling.
#

def plot_spatial_pattern(df, title, output_pdf):
    """Save a spatial point-cloud panel with equal x/y scale."""
    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 6))
    sns.scatterplot(data=df, x="x", y="y", hue="celltype", s=8, linewidth=0, palette="tab10", ax=ax)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.legend(title="Cell type", frameon=False, markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(output_pdf, bbox_inches="tight")
    plt.close(fig)


def plot_matrix(matrix, title, output_pdf, cmap="RdBu_r", center=None, cbar_label="Score"):
    """Save a square heatmap used by the benchmark panels."""
    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(matrix, cmap=cmap, center=center, square=True, linewidths=0.4, cbar_kws={"label": cbar_label}, ax=ax)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(output_pdf, bbox_inches="tight")
    plt.close(fig)


In [ ]:
# Panel mapping:
# - Figure 1g-i, Squidpy comparator helper. Converts the synthetic point table to AnnData and runs Squidpy neighborhood enrichment for the comparator heatmaps.
#

import anndata as ad
import squidpy as sq


def dataframe_to_adata(df):
    """Convert a point table into a minimal AnnData object for neighborhood methods."""
    adata = ad.AnnData(X=np.zeros((df.shape[0], 1), dtype=float))
    adata.obs["celltype"] = pd.Categorical(df["celltype"].to_numpy())
    adata.obsm["spatial"] = df[["x", "y"]].to_numpy()
    adata.obs_names = [f"cell_{i}" for i in range(df.shape[0])]
    return adata


def compute_squidpy_ne(df, n_perms=1000, radius=None, n_neighs=None):
    """Run Squidpy neighborhood enrichment and return its z-score matrix."""
    adata = dataframe_to_adata(df)
    sq.gr.spatial_neighbors(adata, coord_type="generic", spatial_key="spatial", radius=radius, n_neighs=n_neighs)
    sq.gr.nhood_enrichment(adata, cluster_key="celltype", n_perms=n_perms)
    cats = adata.obs["celltype"].cat.categories
    z = adata.uns["celltype_nhood_enrichment"]["zscore"]
    return pd.DataFrame(z, index=cats, columns=cats)


In [ ]:
# Panel mapping:
# - Figure 1g-i, Giotto comparator helper. Wraps Giotto::cellProximityEnrichment through rpy2 so the Giotto benchmark panels can be regenerated from the same point tables.
#

def compute_giotto_matrix(df, n_simulations=1000, k=6):
    """Run Giotto cell proximity enrichment and return a square score matrix.

    The manuscript benchmark used Giotto through the R package interface. This
    wrapper keeps the same data contract as the Python benchmark helpers:
    an input table with x, y, and celltype columns is converted to a minimal
    Giotto object, a kNN spatial network is created, and
    Giotto::cellProximityEnrichment is run on the celltype labels.
    """
    try:
        import rpy2.robjects as ro
        from rpy2.robjects import pandas2ri
        from rpy2.robjects.conversion import localconverter
    except ImportError as exc:
        raise ImportError("Install rpy2 and the R Giotto package to run this cell.") from exc

    r_code = """
    function(point_table, k, n_simulations) {
      suppressPackageStartupMessages(library(Giotto))
      point_table <- as.data.frame(point_table)
      point_table$cell_ID <- as.character(seq_len(nrow(point_table)))
      spatial_locs <- point_table[, c("cell_ID", "x", "y")]
      expr <- matrix(0, nrow = 1, ncol = nrow(point_table))
      rownames(expr) <- "dummy"
      colnames(expr) <- point_table$cell_ID
      g <- createGiottoObject(raw_exprs = expr, spatial_locs = spatial_locs)
      pDataDT(g)$celltype <- as.character(point_table$celltype)
      g <- createSpatialNetwork(gobject = g, method = "kNN", k = as.integer(k), name = "spatial_network")
      enrich <- cellProximityEnrichment(
        gobject = g,
        cluster_column = "celltype",
        spatial_network_name = "spatial_network",
        number_of_simulations = as.integer(n_simulations)
      )
      if ("original" %in% names(enrich)) enrich <- enrich$original

      # Giotto versions can return either a matrix-like object or a long table.
      if (is.matrix(enrich)) return(enrich)
      enrich <- as.data.frame(enrich)
      from_col <- intersect(c("from", "cell_type", "celltype_1", "source"), names(enrich))[1]
      to_col <- intersect(c("to", "neighbor", "celltype_2", "target"), names(enrich))[1]
      score_col <- intersect(c("enrichm", "enrichment", "zscore", "PI_value", "score"), names(enrich))[1]
      if (any(is.na(c(from_col, to_col, score_col)))) {
        stop("Could not identify Giotto enrichment columns in cellProximityEnrichment output.")
      }
      labs <- sort(unique(c(as.character(enrich[[from_col]]), as.character(enrich[[to_col]]))))
      mat <- matrix(NA_real_, nrow = length(labs), ncol = length(labs), dimnames = list(labs, labs))
      for (i in seq_len(nrow(enrich))) {
        mat[as.character(enrich[[from_col]][i]), as.character(enrich[[to_col]][i])] <- as.numeric(enrich[[score_col]][i])
      }
      mat
    }
    """
    r_func = ro.r(r_code)
    r_input = df[["x", "y", "celltype"]].copy()
    r_input["celltype"] = r_input["celltype"].astype(str)
    with localconverter(ro.default_converter + pandas2ri.converter):
        r_df = ro.conversion.py2rpy(r_input)
    r_matrix = r_func(r_df, int(k), int(n_simulations))
    values = np.asarray(r_matrix, dtype=float)
    row_names = list(r_matrix.rownames) if r_matrix.rownames is not None else sorted(r_input["celltype"].unique())
    col_names = list(r_matrix.colnames) if r_matrix.colnames is not None else row_names
    return pd.DataFrame(values, index=row_names, columns=col_names)


In [ ]:
# Panel mapping:
# - Figure 1g-i, ANE comparator helper. Runs the analytical neighborhood enrichment/anhood comparator used beside Squidpy and Giotto.
#

def compute_ane_matrix(df, anhood_repo=Path("/data/taobo.hu/projects/mouse_pup/analytical-enrichment-test")):
    """Run analytical neighborhood enrichment (ANE/anhood) on a point table."""
    import sys
    repo = Path(anhood_repo)
    if repo.exists() and str(repo) not in sys.path:
        sys.path.append(str(repo))
    from anhood.squidpy_wrapper import anhood

    adata = dataframe_to_adata(df)
    sq.gr.spatial_neighbors(adata, coord_type="generic", spatial_key="spatial")
    anhood(adata, cluster_key="celltype")
    cats = adata.obs["celltype"].cat.categories
    z = adata.uns["celltype_nhood_enrichment"]["zscore"]
    return pd.DataFrame(z, index=cats, columns=cats)


In [ ]:
# Panel mapping:
# - Figure 1a-i, panel-generation cell. Creates the modular/nested/random spatial panels, writes the Cell-GPS/COSTE SSS heatmaps, and attempts Squidpy/Giotto/ANE comparator outputs for each pattern.
#

OUTPUT_DIR = Path("outputs/main_figure_1_synthetic_benchmark")
datasets = {
    "modular": make_synthetic_modular(module12_dist=5.0),
    "nested": make_nested_variant(),
    "random": make_synthetic_random(),
}

for name, df in datasets.items():
    plot_spatial_pattern(df, f"Synthetic {name} pattern", OUTPUT_DIR / f"{name}_pattern.pdf")
    row_coph, _ = compute_coste_structuremap(df, f"synthetic_{name}", OUTPUT_DIR)
    plot_matrix(row_coph, f"COSTE SSS - {name}", OUTPUT_DIR / f"COSTE_{name}.pdf", cbar_label="COSTE SSS")
    squidpy_z = compute_squidpy_ne(df, n_perms=1000)
    squidpy_z.to_csv(OUTPUT_DIR / f"Squidpy_NE_{name}.csv")
    plot_matrix(squidpy_z, f"Squidpy NE - {name}", OUTPUT_DIR / f"Squidpy_NE_{name}.pdf", cmap="coolwarm", center=0, cbar_label="z-score")

    # The source benchmark also wrote Giotto and ANE comparator heatmaps.
    # These methods need their original R/Python environments, so missing
    # dependencies are reported without stopping the COSTE/Squidpy outputs.
    for method, func in {"Giotto": compute_giotto_matrix, "ANE": compute_ane_matrix}.items():
        try:
            matrix = func(df)
        except Exception as exc:
            print(f"{method} skipped for {name}: {exc}")
            continue
        matrix.to_csv(OUTPUT_DIR / f"{method}_{name}.csv")
        plot_matrix(matrix, f"{method} - {name}", OUTPUT_DIR / f"{method}_{name}.pdf", cmap="coolwarm", center=0, cbar_label="score")
